# Understanding and Implementing Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks by`(Lewis et al., 2020).`

# 

## 1. Introduction and Motivation

Consider that you are building a question answer but then as you proceed all of sudden you hit the wall. You realise that traditional large language models like GPT-3 or BERT store all their knowledge in their parameters (the weights of the neural network) but they static and do not changes with the world. For example if you will asked your personal chatbot which had knowledge till November 2022 that **When did Lionel Messi win the world cup?** then it will answer that **Messi never won the world cup** or **Maximum Messi has reached closes to world cup is to the final in 2014.** It has no of the events that happened after that duration of time.<br>

Now this approach has several other problems also. For example:

1. **Hallucinations:** Models sometimes sound very knowledgeable and sophisticated but they are giving completely trash information about a certain topic. If you have no idea about that field then there is no way to verify those facts while models are confidently making up the facts.

2. **Knowledge Capacity:** Models have limited amoung of knowledge because we have to fit them into parameters but we can't fit billions of parameters which are needed to store knowledge of whole world and also such systems are difficult to scale.

Now by RAG we combine best of both worlds:
1. **Retriver:** helps us find relevant documents from a huge text corpus.
2. **Generator:** conditions on the query and retrieved documents to produce factual answers.

### 1.1 Problem Which RAG Attempts to Solve

Closed-book generators (like vanilla BART or GPT) must store all facts within parameters but they hallucinate on unseen or conflicting queries. Yet most real world tasks such as QA, fact checking, dialogue and summarization require grounding answers in external knowledge. RAG treats doucments as hidden variables that we conclude from observable and measurable variables through a mathematical model. For example consider a query $x$ and model is retrieving the documents $z_1$, $z_2 ... $, and then marginalizes over them while generating the final answer $y$.

## 2. Architecture of RAG

RAG has **three main components**:
```

Input Query (x)
     │
     ▼
┌─────────────────────────┐
│   RETRIEVER p_η(z|x)    │  ← Finds relevant documents
│(Dense Passage Retriev)  │
└──────────┬──────────────┘
           │
           ▼
     Top-K Documents (z)
           │
           ▼
┌─────────────────────────┐
│  GENERATOR p_θ(y|x,z)   │  ← Generates answer
│       (BART)            │
└──────────┬──────────────┘
           │
           ▼
      Answer (y)

```

### 2.1 Understanding the Retriever
DPR stands for Dense Passage Retrieval which is a neural retrieval system introduced by Facebook AI [Karpukhin et al., 2020](https://arxiv.org/abs/2004.04906). It’s deeply connected to how RAG retrieves documents.

DPR represents both queries and documents as continuous vectors using BERT like encoders and retrieves relevant documents by computing vector similarity (typically inner product) between a query vector and millions of document vectors stored in an index. DPR replaced traditional keyword-based search (like TF-IDF, BM25) with a neural, semantic search that works well for open-domain question answering. The retriever hastwo BERT encoders:

- $\text{Query encoder: } q(x) = f_q(x) \in \mathbb{R}^d$
- $\text{Document encoder: } d(z) = f_d(z) \in \mathbb{R}^d$ 


Our query enocder takes input $x$ and converts it into a dense vector representation $q(x)$. Then our document encoder has already been run on the entire knowledge base (e.g., all of Wikipedia, split into 100-word chunks 16). It creates a vector $d(z)$ for every single document $z$. Finally all these document vectors are stored in a massive Document Index18. The model uses **Maximum Inner Product Search (MIPS)** to instantly find the top-K document vectors $d(z)$ that are closest to our query vector $q(x)$

> In layman terms retriever turns our question into a vector and finds the K Wikipedia chunks whose vectors are the most similar.


### 2.2 Inside the Generator
The generator (BART or T5) models the probability of generating the sequence $y = (y_1, \ldots, y_N)$:<br>
$p_\theta(y \mid x, z)
= \prod_{i=1}^{N} p_\theta(y_i \mid x, z, y_{1:i-1})$

Here we are performing following steps:

* **Input:** concatenate `[x; z]` as the encoder input (query + passage).
* **Output:** autoregressive decoding of (y_i) one token at a time.
* **Training:** teacher forcing — the decoder conditions on the ground-truth prefix $y_{1:i-1}$.
* **Inference:** uses beam search or nucleus sampling for decoding.

To explain in short generator is pre-trained sequence-to-sequence model specifically mostly BART-large. The Generator takes both the original input $x$ and the text of a retrieved document $z$ and simply concatenates them and finally auto-regressively generates the final answer $y$.
